Check runtime

In [ ]:
!nvidia-smi

Clone repo

In [ ]:
!git clone https://github.com/EnsiyeTahaei/DeepAnT-Time-Series-Anomaly-Detection.git

Install dependencies

In [5]:
!pip install -q numpy pandas torch pytorch_lightning scikit-learn matplotlib PyYAML omegaconf

Setup config (bypass argparse which doesn't work in notebooks)

In [ ]:
%cd /content/DeepAnT-Time-Series-Anomaly-Detection

In [ ]:
import os
import random
import logging
import numpy as np
import torch
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load config manually (argparse doesn't work in notebooks)
config = OmegaConf.load("config.yaml")

# Choose your dataset: "air_quality" or "nab"
DATASET = "nab"

cfg = OmegaConf.merge(config.common, config.dataset[DATASET])

# Prepare config
os.makedirs(cfg.run_dir, exist_ok=True)
random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {cfg.device}")
print(OmegaConf.to_yaml(cfg))

Load data

In [ ]:
from utils.data_utils import load_data

train_dataset, val_dataset, test_dataset, feature_dim = load_data(
    cfg.dataset_name, cfg.window_size, cfg.device
)

print(f"Train: {train_dataset.data_x.shape}")
print(f"Val:   {val_dataset.data_x.shape}")
print(f"Test:  {test_dataset.data_x.shape}")
print(f"Features: {feature_dim}")

Train the model

In [ ]:
from deepant.trainer import DeepAnT

model = DeepAnT(cfg, train_dataset, val_dataset, test_dataset, feature_dim)
model.train()

Detect anomalies & visualize

In [ ]:
model.detect_anomaly()

# Display the saved plot inline
from IPython.display import Image, display
display(Image(filename=os.path.join(cfg.run_dir, "anomalies_visualization.png")))

Following README

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python main.py --dataset_name nab